> **Interactive lesson:** Run the code cells, change inputs, and record your observations in the learner cells. This notebook is generated from the [Markdown source](04-llm-training-objectives.md); edit that source and rebuild rather than editing generated cells by hand.


# 1.4 — LLM Training Objectives

**Depth: MASTER**

**Goal:** connect the probability objective, the target tensors, and the optimizer update; recognize training bugs that can make an apparently successful model unusable.

[Month 1 roadmap](../README.md) · [Previous: Tokenization](03-tokenization.ipynb) · [Next: Modern Variants](05-modern-transformer-and-llm-variants.ipynb)


## Lesson 1.4.1 — Autoregressive factorization

> **Read and watch:** [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) connects next-token prediction to broad capabilities. Karpathy's [build GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY) shows how shifted targets become an autoregressive training batch.

For a token sequence `x1, ..., xT`, the chain rule gives:

```text
P(x1, ..., xT) = ∏ P(xt | x1, ..., x(t−1))
```

The model parameterizes these conditionals. A start token or an initial prefix supplies the first context when required by the data convention. Maximizing the training sequences' likelihood is equivalent to minimizing their negative log-likelihood. With one correct token ID per prediction, this is categorical cross-entropy:

```text
loss = −(1/N) Σ log P(correct target token | preceding tokens)
```

`N` is the number of scored, non-ignored target positions, not necessarily the number of sequences. The causal architecture and the shifted targets must agree about what “preceding” means.


## Lesson 1.4.2 — Shift the sequence exactly once

Take the toy ID stream `[8, 3, 6, 2, 9]`. A four-position training example is:

```text
input:    [8, 3, 6, 2]
target:   [3, 6, 2, 9]
position:  0  1  2  3
```

The logit at input position 2 sees the prefix `[8,3,6]` and predicts ID 2. A `[B,T,V]` output supplies `B×T` predictions in one forward pass. The causal mask prevents later input tokens from leaking into earlier predictions.

Some high-level model APIs accept unshifted labels and perform the shift internally. A custom PyTorch loop normally shifts explicitly. Shifting both in the dataset and inside a model skips targets; shifting nowhere trains the wrong task. Write down a five-token example whenever debugging alignment.

The standard tensor path is:


In [ ]:
import torch
import torch.nn.functional as F

stream = torch.tensor([[8, 3, 6, 2, 9]])
inputs, targets = stream[:, :-1], stream[:, 1:]
logits = torch.randn(1, 4, 10, requires_grad=True)
loss = F.cross_entropy(logits.reshape(-1, 10), targets.reshape(-1))
loss.backward()
assert logits.grad.shape == logits.shape


Use `reshape` when slices may be non-contiguous. The flattening must preserve the pairing of each logit row and its target ID.


## Lesson 1.4.3 — Cross-entropy, numerics, and gradients

For one target with assigned probability 0.25, loss is `−ln(0.25) ≈ 1.3863` nats. Giving it probability 0.5 reduces loss to about 0.6931. Probabilities assigned to incorrect tokens affect the normalization and therefore the correct token's probability.

From logits `z`:

```text
loss(z, y) = logsumexp(z) − z[y]
∂loss/∂z[k] = softmax(z)[k] − 1[k=y]
```

The gradient raises the correct logit relative to others during gradient descent. This does not imply each individual parameter always moves in a direction with an obvious linguistic meaning; it participates in many examples and layers.

Pass **raw logits** to [PyTorch cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html). It combines log-softmax and the loss in a numerically stable way. Applying softmax first makes the library treat probabilities as logits, optimizing a different expression. Implementing `log(softmax(z))` naively can also underflow for extreme logits.

Perplexity is `exp(mean token loss)` when loss uses natural logarithms. A uniform distribution over `V` tokens has loss `ln(V)` and perplexity `V`. Perplexity is neither classification accuracy nor a directly comparable score across arbitrary tokenizers and datasets.


## Lesson 1.4.4 — Teacher forcing and the parallel-training distinction

Teacher forcing supplies the true previous tokens during training. Every prefix is already available, so a causal transformer evaluates all target positions in parallel. At generation time, the next prefix contains the model's own sampled tokens:

```text
training:  true prefix → distribution → score true next token
inference: current prefix → distribution → choose token → extend prefix
```

A small mistake can change later contexts during generation. This mismatch is often called exposure bias. Teacher forcing remains a standard, efficient way to optimize autoregressive likelihood; it is not an error to replace every training prefix with samples by default.

Evaluation mode and disabled gradients are separate controls. `model.eval()` changes modules such as dropout. `torch.no_grad()` or inference mode disables gradient tracking. Validation normally needs both. Neither performs a training update.


## Lesson 1.4.5 — Masked language modeling and related objectives

A masked language model sees context on both sides of selected hidden tokens and predicts those tokens. [BERT](https://arxiv.org/abs/1810.04805) is a representative encoder model. Its selected-token corruption includes more than simply replacing every selected token with a mask symbol; distinguish the objective from an oversimplified implementation.

| Objective | Visible information | Scored targets | Natural model pattern |
|---|---|---|---|
| Autoregressive LM | Earlier tokens | Next token at each scored position | Causal decoder |
| Masked LM | Unmasked tokens on both sides | Selected original tokens | Bidirectional encoder |
| Span denoising | Corrupted source and generated target prefix | Missing spans or reconstructed target | Often encoder-decoder |

[T5](https://arxiv.org/abs/1910.10683) provides a concrete text-to-text denoising example. These objectives learn useful representations through different prediction problems. A bidirectional encoder cannot simply be used as an ordinary causal decoder without changing how information flows and how it was trained.


## Lesson 1.4.6 — Why token prediction supports broader capabilities

Predicting realistic continuations rewards representing syntax, entities, discourse, world regularities, code structure, and patterns of problem solving present in data. The model can reuse these internal representations across tasks. This explains why a simple objective can support complex behavior without a separate labeled dataset for each capability.

The objective still rewards predictive fit to a data distribution. It does not directly guarantee truth, robust reasoning, instruction following, or trustworthy uncertainty. A fluent continuation can be wrong. Pretraining, instruction tuning, preference optimization, and application-level evaluation serve different roles; this month establishes the pretraining mechanics before later roadmap modules examine adaptation.


## Lesson 1.4.7 — Loss masking, splits, and reliable validation

For padded targets, mark ignored positions and average only over valid ones. An all-ignored batch has no meaningful mean loss; reject or skip it deliberately. With unequal valid-token counts per batch, aggregate the sum of token losses and divide by the total valid-token count. Averaging batch means gives each batch equal weight regardless of how many predictions it contains.

Separate documents or sources before creating training windows. Randomly splitting overlapping windows leaks near-identical text into validation. Duplicate content can also cross document boundaries, so deduplication and task-relevant evaluation matter.

A healthy tiny-model debugging sequence is: inspect one batch, verify causality, overfit one small training batch, then train with held-out data. Low validation loss with a broken mask is not evidence of language-model quality: the model may be seeing the answer in its input.

The optimizer loop should make every transition visible:

```text
zero gradients → forward → loss → backward → optional gradient clipping
               → optimizer step → record metrics
```

Gradient accumulation intentionally changes when gradients are cleared and when the optimizer steps. Do not accidentally accumulate gradients because `zero_grad()` was forgotten.


## Checkpoint

1. What target aligns with the logit at input index `i` in an explicitly shifted next-token dataset?
2. Why does causal masking still matter when inputs and targets are shifted correctly?
3. Why pass logits directly to cross-entropy?
4. Why does low pretraining loss not prove that the model is a reliable assistant?


<details>
<summary>Show answers</summary>

1. The stream token at `i+1`.
2. That target often also appears as the next input position. Without the mask, earlier logits can attend to it and learn to copy the answer.
3. Cross-entropy implements stable log-softmax plus negative log-likelihood. Passing already normalized probabilities changes the calculation.
4. The objective fits observed continuations. Truthfulness, instruction adherence, domain accuracy, and robust behavior require their own data and evaluation.

</details>


## Hands-on exercises

1. For target probabilities `[0.5, 0.25, 0.125]`, calculate mean loss in nats and perplexity. Derive the logit gradient for a target at index 1 when predicted probabilities are `[0.2, 0.5, 0.3]`.
2. Create shifted inputs/targets from `[4,7,1,8,2,5]` using windows of length 3 at starts 0 and 2. Explain which token each final logit predicts.
3. Batch A has 2 valid tokens with mean loss 1.0; batch B has 6 with mean loss 3.0. Compute the corpus mean and explain the error in averaging the two means.
4. Design two tests that would catch an implementation with suspiciously excellent training loss caused by future-token leakage.


In [ ]:
# Your work here


<details>
<summary>Show exercise solutions</summary>

1. Mean loss is `(ln 2 + ln 4 + ln 8)/3 = ln 4 ≈ 1.386294`; perplexity is 4. The gradient is `[0.2, -0.5, 0.3]`.
2. Start 0: input `[4,7,1]`, target `[7,1,8]`. Start 2: input `[1,8,2]`, target `[8,2,5]`. Final logits predict 8 and 5 respectively.
3. `(2×1 + 6×3)/8 = 2.5`. The unweighted average 2.0 overrepresents the smaller batch.
4. Change only future tokens and verify that earlier logits remain unchanged in evaluation mode. Also inspect attention's strictly upper triangle and assert zero mass there. A loss-only test can miss a leakage bug; these tests target the information-flow invariant.

</details>


## Completion criteria

Build shifted batches, derive token cross-entropy and its gradient, explain teacher forcing versus generation, and validate a training run with causal and data-isolation checks. Carry those checks into the project.


## Primary references

- [PyTorch cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) — input/target and reduction semantics.
- [BERT](https://arxiv.org/abs/1810.04805) — masked pretraining.
- [T5](https://arxiv.org/abs/1910.10683) — text-to-text objectives.
- [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) — capabilities from autoregressive pretraining at scale.
